# 2026 March Madness ML Model Prediction 

## Raw Data

In [6]:
import pandas as pd

teams = pd.read_csv("data/mens/MTeams.csv")
regular_season = pd.read_csv("data/mens/MRegularSeasonDetailedResults.csv")
rankings = pd.read_csv("data/mens/MRankings.csv")
tourney_seeds = pd.read_csv("data/mens/MNCAATourneySeeds.csv")
tourney_results = pd.read_csv("data/mens/MNCAATourneyCompactResults.csv")
prediction_pairs = pd.read_csv("data/mens/MNCAATourneyPredictions.csv")

## Data To Care About

### Season Scope

We will train on seasons **2014-2022 [8]** and test on seasons **2023-2025 [3]**.

Why this scope?
1) It focuses on a more modern era of college basketball (foul baiting, 3-pointer centric). This change occured, I believe, around 2014 when Steph Curry lit up the NBA.
2) It avoids mixing in older seasons that may reflect a meaningfully different style of play (more physical back then).
3) The .csv files give us so much data, and I don't feel like training into the 90s and early 2000s.

**Note**: 2020 season is excluded since there was no NCAA tournament due to COVID-19 (Payton Pritchard I feel for you).

In [5]:
TRAIN_SEASONS = [2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022]
TEST_SEASONS = [2023, 2024, 2025]
MODEL_SEASONS = TRAIN_SEASONS + TEST_SEASONS

# Keep only teams relevant to the seasons we care about.
teams_model = teams[
    (teams["FirstD1Season"] <= max(MODEL_SEASONS))
    & (teams["LastD1Season"] >= min(MODEL_SEASONS))
].copy()

# Season-based subsets for the main modeling data.
regular_season_model = regular_season[regular_season["Season"].isin(MODEL_SEASONS)].copy()
regular_season_train = regular_season[regular_season["Season"].isin(TRAIN_SEASONS)].copy()
regular_season_test = regular_season[regular_season["Season"].isin(TEST_SEASONS)].copy()

rankings_model = rankings[rankings["Season"].isin(MODEL_SEASONS)].copy()
rankings_train = rankings[rankings["Season"].isin(TRAIN_SEASONS)].copy()
rankings_test = rankings[rankings["Season"].isin(TEST_SEASONS)].copy()

tourney_seeds_model = tourney_seeds[tourney_seeds["Season"].isin(MODEL_SEASONS)].copy()
tourney_seeds_train = tourney_seeds[tourney_seeds["Season"].isin(TRAIN_SEASONS)].copy()
tourney_seeds_test = tourney_seeds[tourney_seeds["Season"].isin(TEST_SEASONS)].copy()

tourney_results_model = tourney_results[tourney_results["Season"].isin(MODEL_SEASONS)].copy()
tourney_results_train = tourney_results[tourney_results["Season"].isin(TRAIN_SEASONS)].copy()
tourney_results_test = tourney_results[tourney_results["Season"].isin(TEST_SEASONS)].copy()

# This file is a submission/reference template, not something we split by season.
# We trim it to teams active in our modeling window so it feels more relevant.
model_team_ids = set(teams_model["TeamID"])
prediction_pairs_model = prediction_pairs[
    prediction_pairs["WTeamID"].isin(model_team_ids)
    & prediction_pairs["LTeamID"].isin(model_team_ids)
].copy()

print("Train seasons:", TRAIN_SEASONS)
print("Test seasons:", TEST_SEASONS)
print("\nteams_model:", teams_model.shape)
print("regular_season_train:", regular_season_train.shape)
print("regular_season_test:", regular_season_test.shape)
print("rankings_train:", rankings_train.shape)
print("rankings_test:", rankings_test.shape)
print("tourney_seeds_train:", tourney_seeds_train.shape)
print("tourney_seeds_test:", tourney_seeds_test.shape)
print("tourney_results_train:", tourney_results_train.shape)
print("tourney_results_test:", tourney_results_test.shape)
print("prediction_pairs_model:", prediction_pairs_model.shape)


Train seasons: [2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022]
Test seasons: [2023, 2024, 2025]

teams_model: (367, 4)
regular_season_train: (41548, 34)
regular_season_test: (16850, 34)
rankings_train: (50784, 7)
rankings_test: (20328, 7)
tourney_seeds_train: (544, 3)
tourney_seeds_test: (204, 3)
tourney_results_train: (535, 8)
tourney_results_test: (201, 8)
prediction_pairs_model: (67161, 2)


## Next Step: Feature Engineering

Now that the data is loaded and filtered, the next step is to turn team-level and game-level data into model-ready features. A good place to start is creating season-level team summaries from the regular season, then merging in rankings and tournament seeds so each tournament matchup can be represented numerically for a baseline `scikit-learn` model.